# L100 · SQL AI Functions on AkzoNobel Coatings Data

Welcome to the first lab of the AkzoNobel Agent Bricks workshop. This notebook is the lowest barrier entry point to AI on Databricks: you call large language models directly from SQL, with no agent, no endpoint to deploy, and no machine learning to manage.

Every query here runs against fictional AkzoNobel coatings data (Finance, Supply Chain, and Commercial) that the shared data setup already loaded into Unity Catalog.

### What you will learn
- The one general purpose function, `ai_query`, that runs any prompt over any table.
- The task specific functions that power the no code Agent Bricks types you build next: `ai_classify`, `ai_extract`, `ai_parse_document`, `ai_summarize`.
- `ai_mask` for governance, so personal data never leaks into a prompt.
- `ai_forecast` for time series, which seeds the Forecast Planner use case in the hackathon.

### How this fits the ladder
SQL AI functions are the building blocks. In L200 you call them from agent tools. In L300 the multi domain supervisor composes them. Treat this notebook as your reference for the raw capability.

### A note on how the cells run
Each query is wrapped in `spark.sql(f"...")` so the catalog and model endpoint come from widgets rather than hardcoded names. This keeps the notebook portable across workspaces and makes it run identically in an interactive session and as a scheduled job.

> Note: the data is synthetic. Product names, accounts, and suppliers are invented for the workshop.


## 0. Setup

The two values below are notebook widgets. Change `catalog` if your data setup used a different Unity Catalog, and change `llm_endpoint` to any Foundation Model or external model endpoint you are entitled to.

Run the next cell once. It reads the widgets into Python variables that every later cell interpolates.


In [ ]:
import re
from datetime import date

# Widgets keep the notebook portable across workspaces and catalogs.
dbutils.widgets.text("catalog", "serverless_lakebase_praneeth_catalog", "Unity Catalog")
dbutils.widgets.text("llm_endpoint", "databricks-meta-llama-3-3-70b-instruct", "LLM endpoint")

catalog = dbutils.widgets.get("catalog")
llm_endpoint = dbutils.widgets.get("llm_endpoint")

# These values are interpolated into SQL, so validate them first. Governance
# starts here: never let an unchecked string flow into a query.
if not re.fullmatch(r"[A-Za-z0-9_]+", catalog):
    raise ValueError(f"Unsafe catalog name: {catalog!r}. Use only letters, digits, and underscore.")
if not re.fullmatch(r"[A-Za-z0-9_.-]+", llm_endpoint):
    raise ValueError(f"Unsafe endpoint name: {llm_endpoint!r}.")

print(f"Using catalog:      {catalog}")
print(f"Using LLM endpoint: {llm_endpoint}")

# Discover one safety data sheet PDF from the volume, so the parsing lesson
# never depends on a hardcoded filename.
sds_dir = f"/Volumes/{catalog}/akzo_docs/raw/sds"
sds_path = [f.path.replace("dbfs:", "") for f in dbutils.fs.ls(sds_dir) if f.path.endswith(".pdf")][0]
print(f"Using SDS document: {sds_path}")

# Compute a forecast horizon a few months past the end of the observed series,
# so the forecast lesson stays valid as the data ages.
max_month = spark.sql(f"SELECT MAX(month) AS m FROM {catalog}.akzo_finance.margin_actuals").first()["m"]
_y, _m = max_month.year, max_month.month + 3
_y, _m = _y + (_m - 1) // 12, (_m - 1) % 12 + 1
forecast_horizon = date(_y, _m, 1).isoformat()
print(f"Forecast horizon:   {forecast_horizon}")

display(spark.sql(f"SHOW TABLES IN {catalog}.akzo_finance"))

## 1. `ai_query` — the general purpose function

`ai_query` sends a prompt to a model endpoint and returns the response as a column. It is the Swiss Army knife: anything the task specific functions do, you can also express here with a custom prompt.

Start with a one row smoke test to confirm the endpoint answers.


In [ ]:
display(spark.sql(f"""
SELECT ai_query('{llm_endpoint}', 'Reply with exactly one word: OK') AS probe
"""))

The real power is running a prompt over every row of a table. Here we ask the model to write a one line sales positioning note for each product, grounded in its product line and region. This is batch inference in pure SQL.


In [ ]:
display(spark.sql(f"""
SELECT
  sku,
  product_name,
  product_line,
  region,
  ai_query(
    '{llm_endpoint}',
    CONCAT(
      'In one sentence, write an internal sales positioning note for this AkzoNobel coating. ',
      'Product: ', product_name, '. Line: ', product_line, '. Region: ', region, '.'
    )
  ) AS positioning_note
FROM {catalog}.akzo_finance.products
LIMIT 5
"""))

## 2. `ai_classify` — Text Classification

`ai_classify` sorts text into labels you provide. No training, no examples. This is exactly the capability behind the Text Classification Agent Bricks type.

Here we segment commercial accounts by the market they most likely serve.


In [ ]:
display(spark.sql(f"""
SELECT
  account_name,
  region,
  ai_classify(
    account_name,
    ARRAY('automotive', 'marine', 'architectural', 'industrial', 'aerospace')
  ) AS market_segment
FROM {catalog}.akzo_commercial.accounts
LIMIT 10
"""))

## 3. `ai_extract` — Information Extraction

`ai_extract` pulls named fields out of free text and returns them as structured values. This is the engine behind the Information Extraction Agent Bricks type, useful for safety sheets, contracts, and emails.

Give it the labels you want and it returns the values it finds.


In [ ]:
display(spark.sql(f"""
SELECT ai_extract(
  'Safety Data Sheet: Product Interpon 700, flash point 23 C, contains xylene and titanium dioxide, supplied by AkzoNobel Powder Coatings.',
  ARRAY('product', 'flash_point', 'hazardous_substances', 'supplier')
) AS extracted
"""))

## 4. `ai_parse_document` — Document Parsing

`ai_parse_document` turns a raw PDF into structured content: text, tables, headers, and layout with bounding boxes. It is the first step of any document workflow and powers the Document Parsing Agent Bricks type.

The data setup loaded synthetic safety data sheet PDFs into a Unity Catalog volume. The setup cell discovered one of them into `sds_path`. We read it as binary and parse it. The result is a rich VARIANT value, so we render it as JSON text to inspect it.


In [ ]:
display(spark.sql(f"""
SELECT
  path,
  to_json(ai_parse_document(content)) AS parsed_json
FROM READ_FILES(
  '{sds_path}',
  format => 'binaryFile'
)
"""))

Because `ai_parse_document` returns a VARIANT, we use variant path syntax (`:`) and `variant_explode` to flatten the document elements, keeping the text content with its element type. This is exactly what a Knowledge Assistant would chunk and index.


In [ ]:
display(spark.sql(f"""
WITH parsed AS (
  SELECT ai_parse_document(content) AS doc
  FROM READ_FILES(
    '{sds_path}',
    format => 'binaryFile'
  )
)
SELECT
  el.value:id::int         AS element_id,
  el.value:type::string    AS element_type,
  el.value:content::string AS content
FROM parsed,
LATERAL variant_explode(doc:document:elements) AS el
WHERE el.value:content IS NOT NULL
LIMIT 15
"""))

## 5. `ai_summarize` — condense long text

`ai_summarize` produces a short summary, optionally bounded by a word target. Useful for turning long contracts or analyst notes into a headline a controller can scan.


In [ ]:
display(spark.sql(f"""
SELECT ai_summarize(
  'AkzoNobel gross margin in EMEA declined three points quarter over quarter. The main driver was a sharp rise in titanium dioxide raw material cost, compounded by adverse FX on USD denominated solvent purchases. Volume held flat while list prices lagged cost inflation, compressing the spread.',
  20
) AS summary
"""))

## 6. `ai_mask` — governance before the prompt

Personal data should never flow into a model prompt unmasked. `ai_mask` redacts the entities you name, so you can build a pipeline that is private by construction. This is your first taste of the governance spine that deepens through L200 and L300.


In [ ]:
display(spark.sql(f"""
SELECT ai_mask(
  'Contact buyer Maria Schmidt at maria.schmidt@example.com or +49 170 5551234 about the Interpon order.',
  ARRAY('person', 'email', 'phone')
) AS masked_text
"""))

## 7. `ai_forecast` — time series in one call

`ai_forecast` extrapolates a time series into the future, returning a point forecast and a confidence band. This single function is the seed for the Forecast Planner use case in the hackathon.

It takes a table, a `time_col`, a `value_col`, and a `horizon` expressed as the target end date. We aggregate monthly finance revenue, then forecast past the end of the observed series.

`ai_forecast` is a preview feature. It is enabled on SQL warehouses and on most clusters. If your compute has the preview disabled, the cell below prints a short note instead of failing, and you can run the same query from a SQL warehouse.


In [ ]:
forecast_sql = f"""
WITH monthly_revenue AS (
  SELECT month AS ds, SUM(revenue_eur) AS y
  FROM {catalog}.akzo_finance.margin_actuals
  GROUP BY month
)
SELECT
  ds,
  ROUND(y_forecast) AS revenue_forecast_eur,
  ROUND(y_lower)    AS lower_band_eur,
  ROUND(y_upper)    AS upper_band_eur
FROM ai_forecast(
  TABLE(monthly_revenue),
  horizon   => '{forecast_horizon}',
  time_col  => 'ds',
  value_col => 'y'
)
ORDER BY ds
"""

try:
    display(spark.sql(forecast_sql))
except Exception as e:
    if "AI_FUNCTION_PREVIEW" in str(e) or "preview" in str(e).lower():
        print("ai_forecast is in preview and disabled on this compute.")
        print("Run the query below from a SQL warehouse to see the forecast:\n")
        print(forecast_sql)
    else:
        raise

## Wrap up

You called large language models from SQL alone, with governance and forecasting, all over synthetic coatings data.

| Function | Powers | Used later in |
| --- | --- | --- |
| `ai_query` | any custom prompt, batch inference | every tier |
| `ai_classify` | Text Classification agent | L100 agent types, ticket triage track |
| `ai_extract` | Information Extraction agent | doc extraction track |
| `ai_parse_document` | Document Parsing agent, Knowledge Assistant | doc extraction track |
| `ai_summarize` | response drafting, briefings | commercial and claims tracks |
| `ai_mask` | governance, private prompts | L200 and L300 governance spine |
| `ai_forecast` | Forecast Planner | forecast planner track |

### Next
Open `01_agent_bricks_types.md` to build the no code Agent Bricks agents (Genie, Knowledge Assistant, Extraction, Parsing, Classification) on top of these same tables and functions.
